# Phase 1 AWS Historical Polymarket ETL Model

This notebook documents the Phase 1 historical ETL model in detail. It is meant to be read first as a design document and then used as a lightweight inspection notebook for the data lake outputs.

The model is Polymarket-only. It turns Polygon `OrderFilled` logs and Gamma market metadata into an Athena-queryable star-style dataset in S3:

```text
Gamma markets + Polygon OrderFilled logs
  -> AWS Batch Python ETL
  -> S3 Bronze raw/decoded data
  -> S3 Silver normalized tables
  -> S3 Gold dimensional/fact tables
  -> Glue Catalog
  -> Athena validation and analysis SQL
```

The current implementation intentionally excludes Kalshi, streaming ingestion, orderbook reconstruction, wallet PnL, dbt, Spark, Snowflake, Redshift, and an API service. The goal is a bounded historical backfill that is reliable enough to scale block ranges gradually.

## Executive Summary

The model has four gold tables:

| Table | Grain | Purpose |
|---|---|---|
| `dim_market` | one row per Gamma market | Human-readable market metadata and status |
| `dim_outcome` | one row per CTF outcome token | Maps token IDs to market/outcome semantics |
| `fact_trades` | one row per decoded fill | Atomic historical trade/fill facts from Polygon logs |
| `fact_market_daily` | one row per market/date | Cheap Athena aggregate for volume/count/price monitoring |

The most important join is:

```text
fact_trades.outcome_id = dim_outcome.token_id
dim_outcome.market_id = dim_market.market_id
```

The model treats a CLOB token ID as the most reliable bridge between on-chain fills and Gamma market metadata. Polygon logs tell us which token traded; Gamma tells us what that token means.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from prediction_market.aws_etl.paths import LakePaths
from prediction_market.aws_etl.schemas import (
    DIM_MARKET_COLUMNS,
    DIM_OUTCOME_COLUMNS,
    FACT_MARKET_DAILY_COLUMNS,
    FACT_TRADES_COLUMNS,
)

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

PosixPath('/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market')

## Sources And Raw Inputs

### Gamma API Market Metadata

Gamma provides off-chain market metadata: market ID, question, slug, category, condition ID, binary outcomes, and CLOB token IDs. The ETL uses Gamma in two ways:

1. Catalog fetches for a sample or seed set of markets.
2. Token-targeted fetches for tokens observed in `OrderFilled` logs.

For historical coverage, token-targeted fetches include both open/default and `closed=true` Gamma queries, then paginate with `limit` and `offset`. Large token batches can be split and retried if Gamma returns server errors.

### Polygon `OrderFilled` Logs

Polygon is the primary source for historical fills. Each log is scoped by block range and exchange contract. The decoder supports current CTF Exchange V2 events and older V1 events. The current V2 event carries fields such as `orderHash`, `maker`, `taker`, `side`, `tokenId`, `makerAmountFilled`, `takerAmountFilled`, `fee`, `builder`, and `metadata`.

### Why Both Sources Are Needed

Polygon logs are canonical for fills but do not contain human-readable market questions. Gamma has market semantics but is not a full historical trade ledger. The model joins them through CLOB/CTF token IDs.

## S3 Lake Layout

The lake follows a Bronze/Silver/Gold layout. Bronze keeps raw and minimally decoded source data. Silver keeps normalized intermediate tables. Gold keeps stable dimensional and fact tables for Athena users.

The production-style S3 layout is:

```text
s3://<data-bucket>/
├── bronze/polymarket/
│   ├── markets_raw/run_id=<run_id>/markets.json
│   ├── orderfilled_raw/block_range=<start>-<end>/run_id=<run_id>/orderfilled_logs.json
│   ├── orderfilled_raw/block_range=<start>-<end>/run_id=<run_id>/orderfilled.parquet
│   └── checkpoints/orderfilled_checkpoint.json
├── silver/polymarket/
│   ├── markets_normalized/part-<run_id>.parquet
│   └── trades_normalized/date=<yyyy-mm-dd>/part-<run_id>.parquet
└── gold/polymarket/
    ├── dim_market/part-<run_id>.parquet
    ├── dim_outcome/part-<run_id>.parquet
    ├── fact_trades/date=<yyyy-mm-dd>/part-<run_id>.parquet
    └── fact_market_daily/date=<yyyy-mm-dd>/part-<run_id>.parquet
```

`fact_trades` and `fact_market_daily` are partitioned by `date`. They are intentionally not partitioned by `market_id` or wallet because those would create too many small partitions for early-stage Athena usage.

In [2]:
example_lake_uri = "s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z"
paths = LakePaths(example_lake_uri)

layout = pd.DataFrame(
    [
        {"layer": "bronze", "object": "markets_raw", "path": paths.markets_raw_json("token-markets")},
        {"layer": "bronze", "object": "orderfilled_raw_json", "path": paths.orderfilled_raw_json("fills", 87999343, 88056942)},
        {"layer": "bronze", "object": "orderfilled_raw_parquet", "path": paths.orderfilled_raw_parquet("fills", 87999343, 88056942)},
        {"layer": "bronze", "object": "checkpoint", "path": paths.checkpoint_json()},
        {"layer": "silver", "object": "markets_normalized", "path": paths.silver_markets("token-markets")},
        {"layer": "silver", "object": "trades_normalized", "path": paths.silver_trades_base()},
        {"layer": "gold", "object": "dim_market", "path": paths.dim_market("token-markets")},
        {"layer": "gold", "object": "dim_outcome", "path": paths.dim_outcome("token-markets")},
        {"layer": "gold", "object": "fact_trades", "path": paths.fact_trades_base()},
        {"layer": "gold", "object": "fact_market_daily", "path": paths.fact_market_daily_base()},
    ]
)
layout

,layer,object,path
0,bronze,markets_raw,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/bronze/polym...
1,bronze,orderfilled_raw_json,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/bronze/polym...
2,bronze,orderfilled_raw_parquet,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/bronze/polym...
3,bronze,checkpoint,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/bronze/polym...
4,silver,markets_normalized,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/silver/polym...
5,silver,trades_normalized,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/silver/polym...
6,gold,dim_market,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/gold/polymar...
7,gold,dim_outcome,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/gold/polymar...
8,gold,fact_trades,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/gold/polymar...
9,gold,fact_market_daily,s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z/gold/polymar...


## Pipeline Jobs

The implementation exposes four AWS Batch jobs. They can run independently, but a historical backfill normally runs in this order:

1. `fetch_orderfilled`: fetch Polygon logs by block range and decode each log into an `orderfilled` Parquet file.
2. `fetch_markets`: extract token IDs from decoded fills, query Gamma for matching markets, and write `dim_market` and `dim_outcome`.
3. `normalize_trades`: join decoded fills to outcome tokens, compute normalized amounts/prices, and write `fact_trades`.
4. `build_market_daily`: aggregate `fact_trades` into daily market-level metrics.

The bounded backfill driver fans out `fetch_orderfilled` across block chunks, waits for all chunks, then submits the downstream jobs sequentially. For the one-day validation, normalization and daily aggregation are configured to stream one Parquet file at a time to stay inside the 8GB Batch container memory limit.

In [3]:
lineage = pd.DataFrame(
    [
        {
            "job": "fetch_orderfilled",
            "inputs": "Polygon RPC eth_getLogs by block range and exchange address",
            "outputs": "bronze/polymarket/orderfilled_raw JSON + decoded Parquet",
            "main_keys": "block_number, transaction_hash, log_index, token_id",
        },
        {
            "job": "fetch_markets",
            "inputs": "Gamma markets API, token IDs from orderfilled_raw",
            "outputs": "bronze markets_raw JSON, silver markets_normalized, gold dim_market, gold dim_outcome",
            "main_keys": "market_id, condition_id, token_id",
        },
        {
            "job": "normalize_trades",
            "inputs": "orderfilled_raw, dim_market, dim_outcome",
            "outputs": "silver trades_normalized, gold fact_trades partitioned by date",
            "main_keys": "trade_id, outcome_id, market_id, date",
        },
        {
            "job": "build_market_daily",
            "inputs": "gold fact_trades",
            "outputs": "gold fact_market_daily partitioned by date",
            "main_keys": "market_id, date",
        },
    ]
)
lineage

,job,inputs,outputs,main_keys
0,fetch_orderfilled,Polygon RPC eth_getLogs by block range and exchange address,bronze/polymarket/orderfilled_raw JSON + decoded Parquet,"block_number, transaction_hash, log_index, token_id"
1,fetch_markets,"Gamma markets API, token IDs from orderfilled_raw","bronze markets_raw JSON, silver markets_normalized, gold dim_market, gold dim_outcome","market_id, condition_id, token_id"
2,normalize_trades,"orderfilled_raw, dim_market, dim_outcome","silver trades_normalized, gold fact_trades partitioned by date","trade_id, outcome_id, market_id, date"
3,build_market_daily,gold fact_trades,gold fact_market_daily partitioned by date,"market_id, date"


## Core Modeling Decisions

### Grain

`fact_trades` is modeled at one decoded fill per event log. The unique ID is:

```text
trade_id = transaction_hash || ':' || log_index
```

This is stable for on-chain event logs because a transaction can emit multiple logs and `log_index` distinguishes them.

### Token Mapping

Each Polymarket binary market has two outcome tokens. Gamma exposes these as `clobTokenIds`, which are normalized into `dim_outcome.token_id`. A decoded fill exposes a traded `token_id`; after normalization this becomes `fact_trades.outcome_id`.

### Amount And Price Normalization

The CLOB uses 6-decimal amounts for collateral and token amounts in the current pipeline. The ETL normalizes raw on-chain amounts by `1e6` and computes:

```text
token_amount = outcome token amount filled / 1e6
usd_amount = collateral amount filled / 1e6
price = usd_amount / token_amount
```

The resulting `price` is the traded outcome-token price in dollar units, generally between 0 and 1 for binary outcome tokens.

### Direction

V2 `side` determines whether the maker is buying or selling the event token. The model stores `maker_direction` and `taker_direction` explicitly so downstream analysis does not have to reinterpret event-specific side semantics.

### Daily Aggregate

`fact_market_daily` is a read-optimized aggregate. It is not the source of truth; `fact_trades` is. Daily rows can always be rebuilt from `fact_trades`.

## Gold Table Schemas

The following cells render the schema lists from the codebase. These are the column contracts used by the Parquet writers and Glue/Athena tables.

In [4]:
def schema_frame(table_name: str, columns: list[str], grain: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "table": table_name,
            "ordinal": range(1, len(columns) + 1),
            "column": columns,
            "grain": grain,
        }
    )

schemas = pd.concat(
    [
        schema_frame("dim_market", DIM_MARKET_COLUMNS, "one row per Gamma market"),
        schema_frame("dim_outcome", DIM_OUTCOME_COLUMNS, "one row per market outcome token"),
        schema_frame("fact_trades", FACT_TRADES_COLUMNS, "one row per decoded on-chain fill"),
        schema_frame("fact_market_daily", FACT_MARKET_DAILY_COLUMNS, "one row per market/date"),
    ],
    ignore_index=True,
)
schemas

,table,ordinal,column,grain
0,dim_market,1,market_id,one row per Gamma market
1,dim_market,2,question,one row per Gamma market
2,dim_market,3,slug,one row per Gamma market
3,dim_market,4,category,one row per Gamma market
4,dim_market,5,status,one row per Gamma market
5,dim_market,6,open_time,one row per Gamma market
6,dim_market,7,close_time,one row per Gamma market
7,dim_market,8,resolution_time,one row per Gamma market
8,dim_market,9,condition_id,one row per Gamma market
9,dim_market,10,volume,one row per Gamma market


### `dim_market`

`dim_market` stores market-level metadata from Gamma.

| Column | Meaning |
|---|---|
| `market_id` | Gamma market identifier. Primary key for this dimension. |
| `question` | Human-readable market question. |
| `slug` | Gamma/Polymarket slug. Useful for URLs and search. |
| `category` | Gamma category when available. |
| `status` | Normalized market status: active, closed, archived, inactive. |
| `open_time` | Parsed market start or creation timestamp. |
| `close_time` | Parsed market end timestamp. |
| `resolution_time` | Placeholder for future resolution metadata. |
| `condition_id` | Conditional Tokens Framework condition ID. |
| `volume` | Gamma-reported market volume, used as metadata only. |
| `liquidity` | Gamma-reported liquidity, used as metadata only. |
| `raw_payload_path` | S3 path to source Gamma JSON. |
| `ingested_at` | ETL ingestion timestamp. |

### `dim_outcome`

`dim_outcome` is the bridge between on-chain fills and human market meaning.

| Column | Meaning |
|---|---|
| `outcome_id` | Outcome ID. Currently equal to `token_id`. |
| `market_id` | Parent Gamma market. Foreign key to `dim_market.market_id`. |
| `outcome_name` | Human label such as Yes/No, Over/Under, candidate name, etc. |
| `outcome_index` | Outcome order from Gamma. Binary markets usually have 0 and 1. |
| `token_id` | CLOB/CTF token ID. Joins to `fact_trades.outcome_id`. |
| `side` | Internal side label from parsed Gamma token fields, usually `token1` or `token2`. |
| `ingested_at` | ETL ingestion timestamp. |

A binary market should normally produce two `dim_outcome` rows. Missing token metadata is the main reason an on-chain fill can fail to map back to a market.

### `fact_trades`

`fact_trades` stores the atomic trade/fill history.

| Column | Meaning |
|---|---|
| `trade_id` | `transaction_hash:log_index`, unique event-log fill key. |
| `market_id` | Market ID from token mapping. May be null for unmapped tokens. |
| `outcome_id` | Traded outcome token ID. Joins to `dim_outcome.token_id`. |
| `timestamp` | Block timestamp in epoch seconds. |
| `date` | UTC date derived from `timestamp`. Partition column. |
| `hour` | UTC hour derived from `timestamp`. Useful for intraday checks. |
| `maker`, `taker` | On-chain addresses from the fill event. |
| `price` | `usd_amount / token_amount`. |
| `token_amount` | Normalized outcome-token amount. |
| `usd_amount` | Normalized collateral amount. |
| `maker_direction`, `taker_direction` | Direction after interpreting event side. |
| `transaction_hash` | Polygon transaction hash. |
| `block_number` | Polygon block number. |
| `order_hash` | Polymarket order hash from event topics/data. |
| `fee_amount` | Normalized fee amount when available. |
| `source_type` | Currently `historical_backfill`. |
| `ingested_at` | ETL ingestion timestamp. |
| `raw_payload_path` | Source decoded-fill prefix. |

`fact_trades` is the main analytical source of truth. All aggregates should be reproducible from this table.

### `fact_market_daily`

`fact_market_daily` provides cheap daily market summaries for Athena.

| Column | Meaning |
|---|---|
| `market_id` | Market ID. |
| `date` | UTC date. Partition column. |
| `daily_volume` | Sum of `fact_trades.usd_amount`. |
| `daily_trade_count` | Count of distinct trade IDs after deduplication. |
| `unique_makers` | Distinct maker addresses for market/date. |
| `unique_takers` | Distinct taker addresses for market/date. |
| `avg_price` | Average fill price. |
| `min_price` | Minimum fill price. |
| `max_price` | Maximum fill price. |
| `close_price` | Last observed fill price by timestamp/trade ID within market/date. |

This table is intentionally narrow. It answers common volume and activity questions without scanning all event-level rows.

## Scaling And Memory Model

The bounded one-day validation exposed two important scaling constraints:

1. Some block chunks contain no fills. Empty Parquet files can have no schema, so projected reads must skip schema-empty files safely.
2. A full day of fills can be too large for an 8GB Batch container if normalization or daily aggregation loads everything at once.

The current implementation handles this by streaming file batches:

- `normalize_trades` reads decoded `orderfilled` files in small batches, joins each batch to dimensions, writes partitioned output parts, and maintains a small `seen_trade_ids` set for deduplication.
- `build_market_daily` reads `fact_trades` files in small batches and merges partial aggregate state keyed by `(market_id, date)`.
- The backfill driver submits conservative flags: `--orderfilled-file-batch-size 1` and `--fact-file-batch-size 1`.

This favors reliability over speed for Phase 1. After multiple days pass data-quality checks, batch sizes can be increased carefully to reduce runtime.

## One-Day AWS Validation Snapshot

The latest bounded validation used UTC date `2026-06-06`:

| Item | Value |
|---|---|
| S3 prefix | `s3://prediction-market-dev-699200216006-us-east-1-data/validation/one-day-utc-20260606-20260610T201830Z` |
| Block range | `87999343` to `88056942` |
| ECR image digest | `sha256:a70e1d3e23cb9ad7e4711e20cc03b1b0535a06f87dd5a6883903df732df26a22` |
| Batch result | End-to-end success after streaming fixes |
| Athena table prefix | `v_one_day_utc_20260606_20260610t201830z` |

The validation found only 4 unmatched rows out of 5,750,779 fact trades. Those 4 rows share one token ID that Gamma did not return even when queried directly. That is a residual metadata gap, not a failed ETL join pattern.

In [5]:
validation_results = {
    "fact_trade_count": 5_750_779,
    "distinct_trade_ids": 5_750_779,
    "duplicate_trade_id_count": 0,
    "dim_market_count": 17_564,
    "dim_outcome_count": 35_128,
    "rows_with_market_id": 5_750_775,
    "rows_with_outcome_match": 5_750_775,
    "null_market_id_rows": 4,
    "unmatched_outcomes": 4,
    "market_id_coverage_pct": 99.9999304442059,
    "outcome_match_coverage_pct": 99.9999304442059,
    "daily_trade_count_sum": 5_750_779,
    "daily_volume_sum": 203_243_139.799711,
}

pd.DataFrame([validation_results]).T.rename(columns={0: "value"})

,value
fact_trade_count,5.750779e+06
distinct_trade_ids,5.750779e+06
duplicate_trade_id_count,0.000000e+00
dim_market_count,1.756400e+04
dim_outcome_count,3.512800e+04
rows_with_market_id,5.750775e+06
rows_with_outcome_match,5.750775e+06
null_market_id_rows,4.000000e+00
unmatched_outcomes,4.000000e+00
market_id_coverage_pct,9.999993e+01


In [6]:
storage_summary = pd.DataFrame(
    [
        {"dataset": "orderfilled_raw", "files": 232, "bytes": 8_942_606_002},
        {"dataset": "dim_market", "files": 1, "bytes": 2_674_046},
        {"dataset": "dim_outcome", "files": 1, "bytes": 4_614_791},
        {"dataset": "fact_trades", "files": 114, "bytes": 796_058_669},
        {"dataset": "fact_market_daily", "files": 1, "bytes": 568_234},
    ]
)
storage_summary.assign(mib=lambda frame: (frame["bytes"] / 1024 / 1024).round(2))

,dataset,files,bytes,mib
0,orderfilled_raw,232,8942606002,8528.33
1,dim_market,1,2674046,2.55
2,dim_outcome,1,4614791,4.40
3,fact_trades,114,796058669,759.18
4,fact_market_daily,1,568234,0.54


## Athena Inspection Queries

The cells below include live Athena inspection queries for the validation-specific tables. A notebook-local toggle controls execution:

```python
RUN_AWS_QUERIES = True
```

Set it to `False` if you want to render the notebook without making AWS calls. The old shell environment variable still works too: `RUN_AWS_NOTEBOOK_CELLS=1`.

Use the validation-specific tables to avoid mixing test slices with broader lake tables.


In [7]:
# Notebook-local AWS execution toggle.
# Set to False if you want to render the notebook without running Athena queries.
RUN_AWS_QUERIES = True

ATHENA_DATABASE = "prediction_market_dev"
ATHENA_WORKGROUP = "prediction-market-dev-phase1"
ATHENA_OUTPUT = "s3://prediction-market-dev-699200216006-us-east-1-athena/results/"
TABLE_PREFIX = "v_one_day_utc_20260606_20260610t201830z"

tables = {
    "dim_market": f"{ATHENA_DATABASE}.{TABLE_PREFIX}_dim_market",
    "dim_outcome": f"{ATHENA_DATABASE}.{TABLE_PREFIX}_dim_outcome",
    "fact_trades": f"{ATHENA_DATABASE}.{TABLE_PREFIX}_fact_trades",
    "fact_market_daily": f"{ATHENA_DATABASE}.{TABLE_PREFIX}_fact_market_daily",
}
print(f"RUN_AWS_QUERIES={RUN_AWS_QUERIES}")
tables


RUN_AWS_QUERIES=True


{'dim_market': 'prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_dim_market',
 'dim_outcome': 'prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_dim_outcome',
 'fact_trades': 'prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_fact_trades',
 'fact_market_daily': 'prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_fact_market_daily'}

In [8]:
quality_sql = {
    "row_counts": f"""
SELECT
  (SELECT count(*) FROM {tables['dim_market']}) AS dim_market_count,
  (SELECT count(*) FROM {tables['dim_outcome']}) AS dim_outcome_count,
  (SELECT count(*) FROM {tables['fact_trades']}) AS fact_trade_count,
  (SELECT count(*) FROM {tables['fact_market_daily']}) AS fact_market_daily_count
""".strip(),
    "duplicate_trade_ids": f"""
SELECT trade_id, count(*) AS duplicate_count
FROM {tables['fact_trades']}
GROUP BY trade_id
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 50
""".strip(),
    "mapping_coverage": f"""
SELECT
  count(*) AS fact_trade_count,
  count_if(f.market_id IS NOT NULL AND f.market_id <> '') AS rows_with_market_id,
  count_if(o.token_id IS NOT NULL) AS rows_with_outcome_match,
  100.0 * count_if(f.market_id IS NOT NULL AND f.market_id <> '') / count(*) AS market_id_coverage_pct,
  100.0 * count_if(o.token_id IS NOT NULL) / count(*) AS outcome_match_coverage_pct
FROM {tables['fact_trades']} f
LEFT JOIN {tables['dim_outcome']} o
  ON f.outcome_id = o.token_id
""".strip(),
    "top_daily_markets": f"""
SELECT m.question, d.date, d.daily_volume, d.daily_trade_count, d.close_price
FROM {tables['fact_market_daily']} d
LEFT JOIN {tables['dim_market']} m
  ON d.market_id = m.market_id
ORDER BY d.daily_volume DESC
LIMIT 20
""".strip(),
}

for name, sql in quality_sql.items():
    print(f"\n-- {name}\n{sql};")


-- row_counts
SELECT
  (SELECT count(*) FROM prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_dim_market) AS dim_market_count,
  (SELECT count(*) FROM prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_dim_outcome) AS dim_outcome_count,
  (SELECT count(*) FROM prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_fact_trades) AS fact_trade_count,
  (SELECT count(*) FROM prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_fact_market_daily) AS fact_market_daily_count;

-- duplicate_trade_ids
SELECT trade_id, count(*) AS duplicate_count
FROM prediction_market_dev.v_one_day_utc_20260606_20260610t201830z_fact_trades
GROUP BY trade_id
HAVING count(*) > 1
ORDER BY duplicate_count DESC
LIMIT 50;

-- mapping_coverage
SELECT
  count(*) AS fact_trade_count,
  count_if(f.market_id IS NOT NULL AND f.market_id <> '') AS rows_with_market_id,
  count_if(o.token_id IS NOT NULL) AS rows_with_outcome_match,
  100.0 * count_if(f.market_id IS NOT NULL AND f.marke

In [9]:
def run_athena_query(sql: str) -> pd.DataFrame:
    """Run an Athena query when the notebook toggle or env var enables AWS calls."""
    enabled = bool(globals().get("RUN_AWS_QUERIES", False)) or os.getenv("RUN_AWS_NOTEBOOK_CELLS") == "1"
    if not enabled:
        print("Skipping AWS call. Set RUN_AWS_QUERIES=True in the notebook or RUN_AWS_NOTEBOOK_CELLS=1 in the shell.")
        return pd.DataFrame()

    import time
    import boto3

    client = boto3.client("athena", region_name="us-east-1")
    qid = client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": ATHENA_DATABASE},
        WorkGroup=ATHENA_WORKGROUP,
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
    )["QueryExecutionId"]
    print(f"Athena query submitted: {qid}")
    print(f"Region: us-east-1 | Workgroup: {ATHENA_WORKGROUP}")
    print("In AWS Console, open Athena in us-east-1, switch to this workgroup, then search Query history for the ID above.")

    while True:
        status = client.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if status in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            break
        time.sleep(2)

    if status != "SUCCEEDED":
        raise RuntimeError(f"Athena query {qid} ended with {status}")

    rows = client.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    if not rows:
        return pd.DataFrame()
    headers = [cell.get("VarCharValue", "") for cell in rows[0]["Data"]]
    values = [[cell.get("VarCharValue") for cell in row.get("Data", [])] for row in rows[1:]]
    return pd.DataFrame(values, columns=headers)

run_athena_query(quality_sql["mapping_coverage"])


Athena query submitted: 200e0f68-d4ab-44f6-92c8-207382a01336
Region: us-east-1 | Workgroup: prediction-market-dev-phase1
In AWS Console, open Athena in us-east-1, switch to this workgroup, then search Query history for the ID above.


,fact_trade_count,rows_with_market_id,rows_with_outcome_match,market_id_coverage_pct,outcome_match_coverage_pct
0,5750779,5750775,5750775,99.9999304442059,99.9999304442059


## Local Lake Inspection

For local dry runs, point `LOCAL_LAKE_URI` at a local lake folder such as `data/aws_lake`. The helper below reads local Parquet outputs if they exist. It does not require AWS.

In [10]:
LOCAL_LAKE_URI = PROJECT_ROOT / "data" / "aws_lake"

def local_dataset_counts(base: Path = LOCAL_LAKE_URI) -> pd.DataFrame:
    datasets = {
        "dim_market": base / "gold" / "polymarket" / "dim_market",
        "dim_outcome": base / "gold" / "polymarket" / "dim_outcome",
        "fact_trades": base / "gold" / "polymarket" / "fact_trades",
        "fact_market_daily": base / "gold" / "polymarket" / "fact_market_daily",
    }
    rows = []
    for name, path in datasets.items():
        files = sorted(path.rglob("*.parquet")) if path.exists() else []
        rows.append(
            {
                "dataset": name,
                "path": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
                "parquet_files": len(files),
                "bytes": sum(file.stat().st_size for file in files),
            }
        )
    return pd.DataFrame(rows)

local_dataset_counts()

,dataset,path,parquet_files,bytes
0,dim_market,/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market/data/aws_lake/gold/polymarket/dim_market,0,0
1,dim_outcome,/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market/data/aws_lake/gold/polymarket/dim_outcome,0,0
2,fact_trades,/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market/data/aws_lake/gold/polymarket/fact_trades,0,0
3,fact_market_daily,/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market/data/aws_lake/gold/polymarket/fact_market_daily,0,0


## Operational Runbook

### Build And Push Image

```bash
cd /Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market

docker buildx build --platform linux/amd64 \
  -f docker/Dockerfile \
  -t 699200216006.dkr.ecr.us-east-1.amazonaws.com/prediction-market-dev-etl:latest \
  --push .
```

### Submit A Bounded Backfill

Use the Python backfill driver from the local environment. It submits many `fetch_orderfilled` jobs, waits, then submits `fetch_markets`, `normalize_trades`, and `build_market_daily`.

```bash
prediction-market-backfill \
  --region us-east-1 \
  --job-queue arn:aws:batch:us-east-1:699200216006:job-queue/prediction-market-dev-etl \
  --job-definition arn:aws:batch:us-east-1:699200216006:job-definition/prediction-market-dev-etl:2 \
  --lake-uri s3://prediction-market-dev-699200216006-us-east-1-data/validation/<run_id> \
  --run-id <run_id> \
  --start-block <start_block> \
  --end-block <end_block> \
  --chunk-size 500 \
  --rpc-batch-size 1000 \
  --date-start <yyyy-mm-dd> \
  --date-end <yyyy-mm-dd> \
  --poll-seconds 30 \
  --max-wait-seconds 14400
```

### Create Validation Tables

```bash
prediction-market-athena-validation \
  --region us-east-1 \
  --database prediction_market_dev \
  --work-group prediction-market-dev-phase1 \
  --lake-uri s3://prediction-market-dev-699200216006-us-east-1-data/validation/<run_id> \
  --table-prefix v_<run_id_safe> \
  --replace \
  --checks
```

### Quality Gates Before Scaling

- `duplicate_trade_id_count = 0`
- `market_id_coverage_pct` near 100 percent
- `outcome_match_coverage_pct` near 100 percent
- `fact_trade_count = daily_trade_count_sum`
- Daily volume is nonzero and plausible
- Failed Batch jobs are understood and either retried or fixed
- S3 output paths are isolated by run prefix while validating

## Known Limitations And Next Model Iterations

### Known Limitations

- Gamma can fail to return metadata for rare historical token IDs. The one-day validation had 4 unmatched rows out of 5,750,779 trades.
- The model does not reconstruct orderbooks. It only models fills.
- The model does not yet compute wallet-level positions or PnL.
- `dim_market.resolution_time` is reserved but not yet populated.
- The daily aggregate uses observed trade close, not official market resolution.
- The current conservative streaming settings favor reliability over runtime.

### Recommended Next Iterations

1. Add a small `dim_unmapped_token` or data-quality output for tokens Gamma cannot resolve.
2. Add block-date helper utilities so future backfills can choose UTC dates without manual block lookup.
3. Increase streaming batch sizes gradually after memory metrics are stable.
4. Add run metadata tables for Batch job IDs, image digests, block ranges, and validation results.
5. Add market resolution metadata once a reliable source is selected.
6. Promote validated prefixes from `validation/` into stable production-style S3 prefixes only after quality gates pass.